#Homework 1 - Agentic RAG

## Setting the environment and getting the documents

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [60]:
documents[0:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

#### Q1. How many lesson pages are in the dataset?

In [7]:
len(documents) #72

72

## Indexing and searching

In [ ]:
#### indexing the documents with minsearch - "content" as a text field and "filename" as a keyword field

In [51]:
from minsearch import Index

def build_index(dataset):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(dataset)
    return index

build_index(documents)

In [ ]:
#### Search with the query "How does the agentic loop keep calling the model until it stops?"

#### Q2. What's the filename of the first result?

In [52]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(question)

print(f'there are {len(search_results)} results.')
print(f'filename of the first result is {search_results[0]["filename"]}.')

there are 10 results.
filename of the first result is 01-agentic-rag/lessons/14-agentic-loop.md.


## RAG

In [ ]:
# define a dataclass RAGresponse to store output_text and usage of a RAG call
from dataclasses import dataclass

@dataclass
class RAGResponse:
    answer: str
    usage: object

In [53]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model="gpt-4o-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results)

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()  # ← moved outside the loop

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResponse(
            answer = response.output_text,
            usage = response.usage
        )



In [55]:
question = "How does the agentic loop keep calling the model until it stops?"

index = build_index(documents)

assistant = RAGBase(
    index = index,
    llm_client=openai_client,
    model = "gpt-5.4-mini")

result = assistant.rag(question)

#### Q3. How many input prompts (tokens) did we send to the model for this request?

In [56]:
result.usage.input_tokens

7111

## Chunking

In [ ]:
from gitsource import chunk_documents

# chunks of size 2000, consecutive chunks overlap (step < size)
chunks = chunk_documents(documents, size=2000, step=1000)

#### Q4. How many chunks are there?

In [ ]:
len(chunks)

295

#### Q5. What is the usage of chunks? How many fewer input tokens does the chunked version send?

In [57]:
question = "How does the agentic loop keep calling the model until it stops?"

index_chunks = build_index(chunks)

assistant_chunks = RAGBase(
    index = index_chunks,
    llm_client=openai_client,
    model = "gpt-5.4-mini")

result_chunks = assistant_chunks.rag(question)

result_chunks.usage.input_tokens

print(f'The chunks use {result.usage.input_tokens/result_chunks.usage.input_tokens}x less tokens')

The chunks use 3.0998256320836965x less tokens


## Agentic RAG